# Pausa práctica 4 — VQE encontrando la energía de una molécula real

**Contexto:** acabamos de ver los 5 casos reales, incluyendo AstraZeneca/NVIDIA/IonQ usando VQE + Quantum Monte Carlo para simular una reacción catalítica. Aquí corremos VQE de verdad sobre la molécula más simple posible: **H₂ (hidrógeno molecular)**.

**El objetivo:** encontrar el estado de menor energía (ground state), el mismo principio de "encontrar el mínimo" que vimos en QAOA y en Quantum Annealing, ahora aplicado a química.

In [1]:
!pip install 'qiskit-nature[pyscf]'

In [2]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Molécula de H2 a distancia de enlace de equilibrio
driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g")
problem = driver.run()

mapper = JordanWignerMapper()
hamiltonian = mapper.map(problem.hamiltonian.second_q_op())

print(f"Número de qubits necesarios: {hamiltonian.num_qubits}")
print(f"Energía de repulsión nuclear: {problem.hamiltonian.nuclear_repulsion_energy:.4f} Ha")


Número de qubits necesarios: 4
Energía de repulsión nuclear: 0.7200 Ha


In [3]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Molécula de H2 a distancia de enlace de equilibrio
driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g")
problem = driver.run()

mapper = JordanWignerMapper()
hamiltonian = mapper.map(problem.hamiltonian.second_q_op())

print(f"Número de qubits necesarios: {hamiltonian.num_qubits}")
print(f"Energía de repulsión nuclear: {problem.hamiltonian.nuclear_repulsion_energy:.4f} Ha")


Número de qubits necesarios: 4
Energía de repulsión nuclear: 0.7200 Ha


## Referencia clásica exacta (diagonalización)

Primero, la solución exacta — esto es lo que un método clásico puede calcular para una molécula tan pequeña (recuerda: esto deja de ser posible más allá de ~50-100 átomos activos con correlación fuerte).

In [4]:
!pip install qiskit_algorithms

In [5]:
from qiskit_algorithms import NumPyMinimumEigensolver

exact_solver = NumPyMinimumEigensolver()
exact_result = exact_solver.compute_minimum_eigenvalue(hamiltonian)
exact_energy = exact_result.eigenvalue.real + problem.hamiltonian.nuclear_repulsion_energy

print(f"Energía exacta (clásica): {exact_energy:.6f} Ha")


Energía exacta (clásica): -1.137306 Ha


## Ahora con VQE (el algoritmo cuántico)

In [6]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator as Estimator

ansatz = efficient_su2(hamiltonian.num_qubits, reps=3, entanglement="full")
estimator = Estimator()
optimizer = SLSQP(maxiter=500)

vqe = VQE(estimator, ansatz, optimizer)
vqe_result = vqe.compute_minimum_eigenvalue(hamiltonian)
vqe_energy = vqe_result.eigenvalue.real + problem.hamiltonian.nuclear_repulsion_energy

print(f"Energía VQE (cuántica):   {vqe_energy:.6f} Ha")
print(f"Energía exacta (clásica): {exact_energy:.6f} Ha")
print(f"Diferencia: {abs(vqe_energy - exact_energy)*1000:.4f} mHa")


Energía VQE (cuántica):   -1.116999 Ha
Energía exacta (clásica): -1.137306 Ha
Diferencia: 20.3073 mHa


**Nota:** el "Ha" es Hartree, la unidad de energía estándar en química cuántica. La "precisión química" (chemical accuracy) se considera lograda con una diferencia menor a ~1.6 mHa frente al valor exacto — si el número de arriba está por debajo de eso, VQE encontró la respuesta correcta.

**El puente al caso de AstraZeneca:** esto es exactamente lo mismo que ellos hicieron, pero con una molécula 1000 veces más simple. Ellos usaron 200,000+ circuitos sobre una reacción catalítica real con un flujo híbrido VQE + Quantum Monte Carlo — nosotros acabamos de correr la versión de juguete, en vivo, en menos de un minuto.